# Chapter 7 — Debug the Boundary

**Book alignment:** Debugging AI From First Principles, Chapter 7

**Question this notebook isolates:** `chunk_count(n, size)` ships with 47 green tests and a
phantom page in production. Does a fixed six-cell boundary table (`0, 1, seam-1, seam,
seam+1, large`) turn one failing input into a **failure shape** that names the mechanism —
a fencepost that fires at every exact multiple, not an empty-only special case?

In [ ]:
import math

def chunk_count(n, size):
    """How many pages for n items at page size `size`? (spec: ceil(n/size), 0 for n==0)"""
    return (n // size) + 1                       # the defect: exact multiples get a phantom page

def spec(n, size):
    return math.ceil(n / size) if n else 0

## 1. The suite is green — because every test is a comfortable middle

The 47 tests exercise `paginate` (a correct `range`/slice loop) and a handful of
non-multiple `chunk_count` cases. None lands on an exact multiple, so the phantom page
never shows.

In [ ]:
def paginate(items, size):
    return [items[i:i + size] for i in range(0, len(items), size)]

# paginate itself is correct (this is what the 47 tests cover):
assert paginate(list(range(100)), 10) == [list(range(i, i + 10)) for i in range(0, 100, 10)]
assert paginate([], 10) == []

# and chunk_count agrees with the spec on every NON-multiple middle:
non_multiples = [(37, 10), (8, 3), (53, 10), (7, 4)]
assert all(chunk_count(n, s) == spec(n, s) for n, s in non_multiples)
print("chunk_count on non-multiple middles:", [chunk_count(n, s) for n, s in non_multiples], "  all correct")
print("a suite that never lands on the seam proves nothing about the seam")

## 2. The six-cell boundary table, predictions from the SPEC (not the code)

In [ ]:
SIZE = 10
cells = {"empty": 0, "single": 1, "seam-1": SIZE - 1, "seam": SIZE, "seam+1": SIZE + 1, "large": 10_000}

print(f"{'cell':8} {'n':>6} {'spec':>5} {'got':>5}  {'diverges?':>9}")
shape = []
for name, n in cells.items():
    want, got = spec(n, SIZE), chunk_count(n, SIZE)
    div = got != want
    shape.append("FAIL" if div else "PASS")
    print(f"{name:8} {n:>6} {want:>5} {got:>5}  {'FAIL' if div else 'pass':>9}")

print("\nfailure shape:", " ".join(shape))
# H1 'empty-only' predicts: FAIL PASS PASS PASS PASS PASS
# H2 'fencepost'   predicts: FAIL PASS PASS FAIL PASS FAIL   <- matches
assert shape == ["FAIL", "PASS", "PASS", "FAIL", "PASS", "FAIL"]
print("H2 (fencepost) supported; H1 (empty-only) falsified - seam and large also fail")

## 3. Confirm the mechanism, then fix the RULE (not the instance)

In [ ]:
# why it fails exactly at multiples: (n // size) + 1 assumes integer division threw away
# a non-zero remainder. True across every regime interior; false when the remainder is 0.
for n in (9, 10, 11, 19, 20, 21):
    print(f"n={n:>2}  n//size={n // SIZE}  remainder={n % SIZE}  buggy={chunk_count(n, SIZE)}  spec={spec(n, SIZE)}")

def chunk_count_fixed(n, size):
    return (n + size - 1) // size            # ceiling division: rounds up only on a real remainder

assert all(chunk_count_fixed(n, SIZE) == spec(n, SIZE) for n in list(cells.values()) + [0, 30, 99, 100])
# an H1-style instance patch (if n == 0: return 0) would leave n=10, n=20 broken:
h1_patch = lambda n, s: 0 if n == 0 else (n // s) + 1
assert h1_patch(10, SIZE) != spec(10, SIZE)
print("\nceiling form passes the whole table; the zero-only patch still breaks every other multiple")

## 4. Metamorphic relations as an oracle where the spec is silent

In [ ]:
import random
rng = random.Random(0)

def check_relations(cc):
    for _ in range(2000):
        k, size = rng.randint(1, 500), rng.randint(1, 500)
        n = rng.randint(0, 5000)
        assert cc(k * size, size) == k, ("exact multiple", k, size)       # cc(k*size, size) == k
        assert cc(n + 1, size) >= cc(n, size), ("monotone", n, size)      # non-decreasing
        assert cc(n + 1, size) - cc(n, size) in (0, 1), ("unit step", n, size)

try:
    check_relations(chunk_count)
    print("buggy chunk_count satisfied the relations?!")
except AssertionError as e:
    print("buggy chunk_count violates:", e.args[0])
check_relations(chunk_count_fixed)
print("fixed chunk_count satisfies every metamorphic relation - convicted the arithmetic with no absolute oracle")

## What we earned

A green suite of middles said nothing about the seam. The six-cell table turned "fails at
n=0" into the shape `FAIL PASS PASS FAIL PASS FAIL` — the exact multiples — which falsifies
the empty-only hypothesis and names a fencepost. The fix is the ceiling rule
`(n + size - 1) // size`, verified by re-running the whole table; an `if n == 0` patch would
have left every other multiple broken. Where the spec is silent, three metamorphic
relations convict the arithmetic anyway.

**Notebook 08 / Chapter 8** converts findings like this into tripwires that fire on their
own: assertions, guards, and contracts at the handoff.